In [2]:
from pynq import Overlay
import numpy as np

ol = Overlay('/home/xilinx/angus.bit')
print(ol.ip_dict.keys())  # should show your IP blocks

dict_keys(['top_0', 'axi_dma_0', 'processing_system7_0'])


In [3]:
# Get handle to your register block
regs = ol.top_0  # name may vary, check ol.ip_dict

# Write and read back a register to prove AXI works
# Write to CONTROL register (offset 0x00)
regs.write(0x00, 0x01)  # enable
val = regs.read(0x00)
print(f"Control: 0x{val:08X}")  # should read back 0x01

# Read STATUS register (offset 0x24)
status = regs.read(0x24)
print(f"Status: 0x{status:08X}")
sync_state = status & 0x7
print(f"Sync state: {sync_state}")  # should be 0 (UNSYNC) with no crank signal

Control: 0x00000001
Status: 0x00000000
Sync state: 0


In [4]:
# Write PLL config registers with sensible defaults
regs.write(0x04, 0xC0)   # GAP_THRESH = 3 missing teeth
regs.write(0x08, 0x0100) # KP
regs.write(0x0C, 0x0010) # KI
regs.write(0x10, 0x0400) # MAX_CORRECTION
regs.write(0x14, 3600)   # PHASE_ANGLE (180 degrees = 3600 steps)
regs.write(0x18, 15)     # PHASE_TOL
regs.write(0x1C, 0)      # TDC_OFFSET
regs.write(0x20, 1)      # DECIMATION
regs.write(0x00, 0x03)   # CONTROL enable

print("Registers written")

Registers written


In [5]:
import time

def read_status():
    status     = regs.read(0x24)
    sync_state = status & 0x7
    raw_angle  = regs.read(0x38)
    crank_angle= regs.read(0x3C)
    eng_angle  = regs.read(0x40)
    sync_loss  = regs.read(0x28)
    
    states = {0:'UNSYNC', 1:'FIRST_GAP', 2:'SYNC_CRANK', 3:'SYNC_FULL'}
    print(f"Sync: {states.get(sync_state,'?'):12s} "
          f"Raw: {raw_angle:4d}  "
          f"Crank: {crank_angle:4d}  "
          f"Engine: {eng_angle:4d}  "
          f"Losses: {sync_loss}")

# Poll continuously
i = 0
while i < 10:
    i = i + 1
    read_status()
    time.sleep(0.1)

Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0
Sync: UNSYNC       Raw:    0  Crank:    0  Engine:    0  Losses: 0


In [9]:
status = regs.read(0x24)
signal_present = (status >> 4) & 0x1
sync_state     = status & 0x7
print(f"signal_present={signal_present} sync_state={sync_state}")

signal_present=1 sync_state=2


In [10]:
ab_count    = regs.read(0x44)
tooth_per   = regs.read(0x48)
gap_per     = regs.read(0x4C)
nco_inc     = regs.read(0x50)
phase_err   = regs.read(0x54)
correction  = regs.read(0x58)

print(f"ab_count    = {ab_count}")
print(f"tooth_per   = {tooth_per}  ({tooth_per/100000:.1f}ms  = {60000/(tooth_per/100000*60):.0f} RPM)")
print(f"gap_per     = {gap_per}  ({gap_per/100000:.1f}ms)")
print(f"nco_inc     = {nco_inc}")
print(f"phase_err   = {phase_err}")
print(f"correction  = {correction}")

ab_count    = 51
tooth_per   = 200122  (2.0ms  = 500 RPM)
gap_per     = 603706  (6.0ms)
nco_inc     = 357
phase_err   = 16636476
correction  = 1024


In [11]:
cam_angle = regs.read(0x5C)
print(f"cam_angle = {cam_angle}  ({cam_angle/10:.1f} degrees)")

cam_angle = 1716  (171.6 degrees)


In [12]:
regs.write(0x14, 1717)   # PHASE_ANG = 171.7 degrees (cam_angle value)
regs.write(0x18, 300)    # PHASE_TOL = 30 degrees tolerance

In [13]:
status = regs.read(0x24)
print(f"sync_state = {status & 0x7}")  # expecting 3 = SYNC_FULL

sync_state = 3


In [16]:
nco_inc     = regs.read(0x50)
phase_err   = regs.read(0x54)
correction  = regs.read(0x58)

print(f"nco_inc     = {nco_inc}")
print(f"phase_err   = {phase_err}")
print(f"correction  = {correction}")

nco_inc     = 357
phase_err   = 21380615
correction  = 1024


In [17]:
# Current: edge_select=1, try edge_select=0
regs.write(0x00, 0x00)  # edge_select = 0
time.sleep(0.5)
status = regs.read(0x24)
print(f"edge_select=0: sync={status & 0x7}")

# Also clear any fault and try again
regs.write(0x00, 0x02)  # fault_clear=1
time.sleep(0.1)
regs.write(0x00, 0x00)  # fault_clear back to 0
time.sleep(0.5)
status = regs.read(0x24)
print(f"After fault clear: sync={status & 0x7}")

edge_select=0: sync=3
After fault clear: sync=3


In [18]:
status = regs.read(0x24)
signal_present = (status >> 4) & 0x1
sync_state     = status & 0x7
print(f"signal_present={signal_present} sync_state={sync_state}")

signal_present=1 sync_state=3


In [19]:
import time

def read_status():
    status     = regs.read(0x24)
    sync_state = status & 0x7
    raw_angle  = regs.read(0x38)
    crank_angle= regs.read(0x3C)
    eng_angle  = regs.read(0x40)
    sync_loss  = regs.read(0x28)
    
    states = {0:'UNSYNC', 1:'FIRST_GAP', 2:'SYNC_CRANK', 3:'SYNC_FULL'}
    print(f"Sync: {states.get(sync_state,'?'):12s} "
          f"Raw: {raw_angle:4d}  "
          f"Crank: {crank_angle:4d}  "
          f"Engine: {eng_angle:4d}  "
          f"Losses: {sync_loss}")

# Poll continuously
i = 0
while i < 10:
    i = i + 1
    read_status()
    time.sleep(0.1)

Sync: SYNC_FULL    Raw: 1091  Crank: 1093  Engine: 1094  Losses: 0
Sync: SYNC_FULL    Raw:   34  Crank:   34  Engine:   35  Losses: 0
Sync: SYNC_FULL    Raw: 6096  Crank: 6097  Engine: 6098  Losses: 0
Sync: SYNC_FULL    Raw: 4932  Crank: 4933  Engine: 4934  Losses: 0
Sync: SYNC_FULL    Raw: 3756  Crank: 3758  Engine: 3759  Losses: 0
Sync: SYNC_FULL    Raw: 2582  Crank: 2583  Engine: 2584  Losses: 0
Sync: SYNC_FULL    Raw: 1422  Crank: 1423  Engine: 1424  Losses: 0
Sync: SYNC_FULL    Raw:  244  Crank:  245  Engine:  247  Losses: 0
Sync: SYNC_FULL    Raw: 6257  Crank: 6259  Engine: 6260  Losses: 0
Sync: SYNC_FULL    Raw: 5098  Crank: 5099  Engine: 5100  Losses: 0


In [20]:
status = regs.read(0x24)
print(f"signal_present = {(status >> 4) & 0x1}")
print(f"sync_state     = {status & 0x7}")
print(f"synced         = {(status >> 5) & 0x1}")

signal_present = 1
sync_state     = 3
synced         = 1


In [21]:
gap_thresh = regs.read(0x04)
print(f"gap_thresh = {gap_thresh}")

gap_thresh = 192
